# RSNA Knee — efficiency submission

**exp008**: DINOv2 ViT-S/14, T=16 slices, 320px, distilled (KL+BCE mix against the 3-member ensemble's soft predictions + `labels_v3_candidate`) — CLAUDE.md §8 efficiency-track model. Accepted 2026-09-10: OOF macro vs `labels_v2_final` 0.8708, only -0.0036 off the full ensemble (0.8744), well inside the 0.015 target.

**fp16** (not bf16 — smaller/older GPUs common in the efficiency lane support fp16 more broadly; exp008's small ViT-S doesn't need bf16's extra dynamic range the way the bigger ensemble members might). **No TTA** (rejected for the ensemble on OOF grounds; not re-tested for exp008, but there's no reason to expect a different verdict at a different resolution — not worth the 2x cost either way for the efficiency lane specifically). **Batched decode**: DICOM decode (`decode_one_for_pool`/`decode_study_slots`, pydicom-based, CPU-bound) is parallelized across worker THREADS (not processes — see cell-1 for the 2026-09-10 finding that a process pool deadlocks after CUDA init and can't locate the worker function through this notebook's custom package loader; threads sidestep both and still parallelize the GIL-releasing C-extension work: DICOM decompression and numpy/scipy resampling), processed in chunks so we're never holding more than one chunk's volumes in memory at once — decode is very likely the dominant cost at this small a model, per CLAUDE.md §8's own note to profile it separately.

**Fold count is a ladder rung, not hardcoded** (EXPERIMENTS.md 2026-09-10): on gold-58 — the only set where every fold's model predicts the same studies — averaging over ALL subsets of each size (not just the best one) gives mean deltas vs the full 5-fold ensemble of -0.0060 (1-fold), -0.0030 (2), -0.0014 (3), -0.0004 (4). The cost of dropping folds is small and gradual, not a cliff, so the notebook defaults to all 5 (best expected accuracy) and only drops folds if a calibration pass shows the time budget genuinely needs it.

**Runtime is a PROJECTION** from a calibration pass on the first studies — not yet backed by a real 1300-study GPU run.

Expects attached Kaggle Datasets: `rsna-knee-code`, `rsna-knee-exp008-weights`, plus the competition data. Internet OFF.

In [ ]:
import importlib.util
import os, sys, time, traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import torch

def _find_dir(root: str, name_contains: str) -> Path | None:
    for p in Path(root).rglob("*"):
        if p.is_dir() and name_contains in p.name:
            return p
    return None

CODE_DIR = _find_dir("/kaggle/input", "rsna-knee-code")
WEIGHTS_DIR = _find_dir("/kaggle/input", "rsna-knee-exp008-weights")
TEST_DIR = _find_dir("/kaggle/input", "rsna-knee-abnormality-detection")
OUT_PATH = Path("/kaggle/working/submission.csv")

print("CODE_DIR:", CODE_DIR)
print("WEIGHTS_DIR:", WEIGHTS_DIR)
print("TEST_DIR:", TEST_DIR)
assert CODE_DIR is not None, "Attach rsna-knee-code"
assert WEIGHTS_DIR is not None and WEIGHTS_DIR.exists(), "Attach rsna-knee-exp008-weights"
assert TEST_DIR is not None, "Attach the competition data"

init_files = list(CODE_DIR.rglob("__init__.py"))
assert init_files, f"No __init__.py found under {CODE_DIR}"
pkg_dir = init_files[0].parent
spec = importlib.util.spec_from_file_location("rsna_knee", pkg_dir / "__init__.py", submodule_search_locations=[str(pkg_dir)])
rsna_knee = importlib.util.module_from_spec(spec)
sys.modules["rsna_knee"] = rsna_knee
spec.loader.exec_module(rsna_knee)

from rsna_knee.dataset import TARGET_COLUMNS, _patch_size_for_backbone
from rsna_knee.infer_core import (
    WallClockGuard, decode_one_for_pool, nearest_patch_multiple, prior_row, run_study_inference,
)
from rsna_knee.model import SlotModel

# Batched decode uses ThreadPoolExecutor, not a process pool (2026-09-10 finding): a process pool
# hit two compounding problems here — forking after CUDA is initialized in the parent deadlocks
# the child, and even a spawn context can't find decode_one_for_pool because the custom
# sys.modules-injection package loader above isn't visible to a freshly-spawned interpreter.
# Threads sidestep both (shared process, no pickling) and still parallelize the expensive part of
# decode, since DICOM decompression (pylibjpeg/gdcm) and numpy/scipy resampling are C extensions
# that release the GIL. decode_one_for_pool itself still lives in the package (not this notebook)
# so it stays usable from either executor type without further changes.

print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())

In [ ]:
BACKBONE = "vit_small_patch14_dinov2.lvd142m"
SLICES_PER_SERIES = 16
IMAGE_SIZE = 320
OUT_SIZE = nearest_patch_multiple(IMAGE_SIZE, _patch_size_for_backbone(BACKBONE))  # 322, patch14 rounding

AUTOCAST_DTYPE = torch.float16  # fp16, not bf16 — see markdown
DECODE_CHUNK_SIZE = 24          # studies decoded per batch
# os.cpu_count() can report a shared host's FULL core count rather than this container's actual
# allocation in some environments (found 2026-09-10 on HPC SLURM: requested 16 cores, cpu_count()
# reported 192) — capped defensively rather than trusted blindly. Threads (not processes, see
# cell-1) make an overshoot here far cheaper than it would be with a process pool, but the cap
# costs nothing regardless.
DECODE_WORKERS = max(1, min((os.cpu_count() or 4) - 1, 32))

BUDGET_SECONDS = 45 * 60  # CLAUDE.md §8 target: <=45min for ~1300 studies
CALIBRATION_STUDIES = 10

DEVICE = "cpu"
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    cap = major * 10 + minor
    supported_caps = [int(a[3:]) for a in torch.cuda.get_arch_list() if a.startswith("sm_")]
    if supported_caps and cap >= min(supported_caps):
        DEVICE = "cuda"
    else:
        print(f"  [WARN] GPU {torch.cuda.get_device_name(0)} (sm_{cap}) unsupported by this torch "
              f"build (min sm_{min(supported_caps) if supported_caps else '?'}) — CPU fallback.")
print("device:", DEVICE, "out_size:", OUT_SIZE, "decode_workers:", DECODE_WORKERS)

In [ ]:
def load_fold_models(weights_dir: Path, backbone: str, device: str, n_folds: int) -> list:
    models = []
    for fold_dir in sorted(weights_dir.glob("fold*"))[:n_folds]:
        ckpt_path = fold_dir / "best.pt"
        if not ckpt_path.exists():
            print(f"  [WARN] no best.pt in {fold_dir}")
            continue
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model = SlotModel(backbone, num_targets=len(TARGET_COLUMNS), pretrained=False).to(device)
        model.load_state_dict(ckpt["ema"])
        model.eval()
        models.append(model)
    return models

ALL_MODELS = load_fold_models(WEIGHTS_DIR, BACKBONE, DEVICE, n_folds=5)
print(f"Loaded {len(ALL_MODELS)}/5 folds.")
if not ALL_MODELS:
    raise RuntimeError("No fold models loaded — cannot run inference.")

In [ ]:
test_df = pd.read_csv(TEST_DIR / "test.csv")
series_df = pd.read_csv(TEST_DIR / "test_series.csv").rename(
    columns={"StudyInstanceUID": "study_uid", "SeriesInstanceUID": "series_uid"}
)
series_root = TEST_DIR / "test_series"
print(f"{len(test_df)} test studies, {len(series_df)} series.")

In [ ]:
def decode_chunk(study_uids: list[str]) -> dict[str, tuple[dict, list, str | None]]:
    """Decodes a chunk of studies in parallel worker THREADS (see cell-1 for why not processes).
    Returns {study_uid: (slots, tag_sides, error_or_None)} — one chunk's volumes are held in
    memory at a time (T=16/320px is small per study, but 1300 studies x 6 slots at once would not
    be)."""
    tasks = [
        (uid, series_df[series_df["study_uid"] == uid].to_dict("records"), str(series_root), SLICES_PER_SERIES, OUT_SIZE)
        for uid in study_uids
    ]
    results = {}
    with ThreadPoolExecutor(max_workers=DECODE_WORKERS) as pool:
        futures = {pool.submit(decode_one_for_pool, t): t[0] for t in tasks}
        for fut in as_completed(futures):
            uid, slots, tag_sides, err = fut.result()
            results[uid] = (slots, tag_sides, err)
    return results

In [ ]:
# Ladder rungs, cheapest-fallback last — fold count is the ONLY lever here (no ensemble members,
# no TTA), per the 2026-09-10 decision not to hardcode a fixed fold count.
LADDER_FOLD_COUNTS = [5, 4, 3, 2, 1]

guard = WallClockGuard(BUDGET_SECONDS)

# --- Calibration: decode+infer the first studies at full (5-fold) fidelity, measure real
# per-study cost, project each rung, pick the highest one (most folds) that fits with margin.
n_calib = min(CALIBRATION_STUDIES, len(test_df))
calib_uids = list(test_df["StudyInstanceUID"].iloc[:n_calib])
calib_t0 = time.time()
calib_decoded = decode_chunk(calib_uids)
calib_rows = []
for uid in calib_uids:
    slots, tag_sides, err = calib_decoded[uid]
    if slots and err is None:
        probs = run_study_inference(slots, tag_sides, ALL_MODELS, DEVICE, SLICES_PER_SERIES, OUT_SIZE, AUTOCAST_DTYPE)
        calib_rows.append({"StudyInstanceUID": uid, **{c: float(p) for c, p in zip(TARGET_COLUMNS, probs)}})
    else:
        print(f"  [WARN] calibration decode failed for {uid}: {err}")
calib_elapsed = time.time() - calib_t0
per_study_at_5fold = calib_elapsed / max(n_calib, 1)
print(f"Calibration: {n_calib} studies, {calib_elapsed:.1f}s ({per_study_at_5fold:.2f}s/study at 5 folds)")

chosen_n_folds = LADDER_FOLD_COUNTS[-1]
for n_folds in LADDER_FOLD_COUNTS:
    # Inference cost scales ~linearly with fold count; decode cost is fold-independent (same
    # decode regardless of how many models consume it), so only discount the inference share.
    # Rough split: assume half the calibrated per-study time is decode, half is 5-fold inference.
    projected_per_study = per_study_at_5fold * 0.5 + (per_study_at_5fold * 0.5) * (n_folds / 5)
    projected_total = projected_per_study * len(test_df)
    fits = projected_total <= guard.budget * 0.85
    print(f"  {n_folds} folds: projected {projected_total / 60:.1f} min for {len(test_df)} studies "
          f"-> {'FITS' if fits else 'too slow'}")
    if fits:
        chosen_n_folds = n_folds
        break

ACTIVE_MODELS = ALL_MODELS[:chosen_n_folds]
print(f"Chosen fold count: {chosen_n_folds}/5")

In [ ]:
rows = list(calib_rows) if chosen_n_folds == 5 else []  # reuse calibration only if it matches the chosen rung
done_uids = {r["StudyInstanceUID"] for r in rows}
remaining_uids = [u for u in test_df["StudyInstanceUID"] if u not in done_uids]

n_prior_fallback = n_dual_flip = 0
for chunk_start in range(0, len(remaining_uids), DECODE_CHUNK_SIZE):
    chunk_uids = remaining_uids[chunk_start:chunk_start + DECODE_CHUNK_SIZE]
    if guard.over(margin=0.05):
        print(f"  [BUDGET] {guard.elapsed():.0f}s used, {len(rows)}/{len(test_df)} done — priors for the rest.")
        for uid in chunk_uids + remaining_uids[chunk_start + DECODE_CHUNK_SIZE:]:
            row = {"StudyInstanceUID": uid, **{c: float(p) for c, p in zip(TARGET_COLUMNS, prior_row())}}
            rows.append(row)
            n_prior_fallback += 1
        break

    decoded = decode_chunk(chunk_uids)
    for uid in chunk_uids:
        row = {"StudyInstanceUID": uid}
        try:
            slots, tag_sides, err = decoded[uid]
            if err is not None or not slots:
                raise RuntimeError(err or "no series decoded successfully for this study")
            if all(s not in ("R", "L") for s in tag_sides):
                n_dual_flip += 1
            probs = run_study_inference(slots, tag_sides, ACTIVE_MODELS, DEVICE, SLICES_PER_SERIES, OUT_SIZE, AUTOCAST_DTYPE)
            row.update({c: float(p) for c, p in zip(TARGET_COLUMNS, probs)})
        except Exception as e:
            print(f"  [WARN] {uid}: {type(e).__name__}: {e} — using label priors")
            n_prior_fallback += 1
            row.update({c: float(p) for c, p in zip(TARGET_COLUMNS, prior_row())})
        rows.append(row)

    print(f"  {len(rows)}/{len(test_df)} studies, {guard.elapsed():.0f}s elapsed ({guard.fraction_used():.1%} of budget)")

print(f"Done: {len(rows)} studies, fold_count={chosen_n_folds}, {n_prior_fallback} full-prior fallbacks, "
      f"{n_dual_flip} dual-flip, {guard.elapsed():.0f}s total.")

In [ ]:
sub_df = pd.DataFrame(rows)[["StudyInstanceUID"] + TARGET_COLUMNS]

assert len(sub_df) == len(test_df), f"row count mismatch: {len(sub_df)} vs {len(test_df)}"
assert set(sub_df["StudyInstanceUID"]) == set(test_df["StudyInstanceUID"]), "study UID mismatch"
assert not sub_df[TARGET_COLUMNS].isna().any().any(), "NaNs in submission"
assert (sub_df[TARGET_COLUMNS] >= 0).all().all() and (sub_df[TARGET_COLUMNS] <= 1).all().all(), "probabilities out of [0,1]"
assert list(sub_df.columns) == ["StudyInstanceUID"] + TARGET_COLUMNS, f"header mismatch: {list(sub_df.columns)}"

sub_df.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH}: {len(sub_df)} rows, all checks passed. Folds used: {chosen_n_folds}/5.")
sub_df.head()